## Positional Encoding

Attention は単語の順番を区別できない(各トークンを並列に扱うため)。そこで「各トークンが系列の何番目か」という位置情報を、埋め込みに足して与える。

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- $pos$: トークンの位置(0, 1, …, seq-1)
- $i$: 次元のインデックス。偶数次元は sin、奇数次元は cos




In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        #箱を作る
        pe = torch.zeros(max_seq, d_model) 
        
        #トークンの位置（縦）
        position = torch.arange(0, max_seq, dtype=torch.float).unsqueeze(1) # .umsqueeze(1):(max_len, 1) の縦ベクトルにする。
        
        # 分母の計算: 10000^(2i/d_model) = exp(2i * log(10000) / d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term) # 最初に作った箱peの偶数次元に入れ込む
        pe[:, 1::2] = torch.cos(position * div_term) # 奇数次元

        pe = pe.unsqueeze(0) # (1, max_len, d_model)　　#0番目に次元を追加　入力xはバッチ次元もあるからやらないと足し算できない

        #位置エンコーディングは固定値だから学習させたくない
        self.register_buffer('pe', pe) #バッファとしてモジュールに登録する　学習対象にならない(勾配刑されない)　self.peで呼び出せるようにしてる
    

    def forward(self, x):  # x: (batch, seq, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

    
